# ASL: Combine Body Position and Local Hand Shape

Select your Kaggle remote Python kernel and run the cells in order. This notebook is self-contained.

**Hypothesis:** body-relative coordinates preserve signing location, while independently normalized hand coordinates make local hand shape easier to learn. This experiment concatenates both representations. It uses the same signer split, seed, optimizer, batch size, maximum epochs, CNN filters, and kernel sizes as notebook 03.

The input increases from **126 to 252 features per frame**, so the first convolution has more parameters. This is an input-representation experiment with a capacity difference, not a parameter-matched ablation. A gain alone cannot isolate which factor caused it.

The default is full-data training for up to 20 epochs. For a pipeline check, set `SAMPLE_LIMIT = 1000` and `EPOCHS = 2`. Confirm Kaggle GPU availability before a full run. Preserve notebook 03 artifacts so the optional comparison can use the current best baseline.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import math
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
SAMPLE_LIMIT = None  # Full dataset; set to 1000 only for a quick pipeline check.
EPOCHS = 20         # Maximum epochs; early stopping may finish sooner.
N_FRAMES = 32
N_FEATURES = 252
BATCH_SIZE = 64
VALIDATION_FRACTION = 0.2
DATA_DIR = None     # Or Path('/kaggle/input/competitions/asl-signs').
BASELINE_RUN_DIR = None  # Optionally Path('/kaggle/working/asl-project/cnn_body_reference_<run_id>').
INPUT_DIR = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/asl-project')

if not INPUT_DIR.is_dir():
    raise RuntimeError('Select a Kaggle remote kernel; /kaggle/input is unavailable.')
if SAMPLE_LIMIT is not None and SAMPLE_LIMIT < 2:
    raise ValueError('SAMPLE_LIMIT must be None or at least 2.')
tf.keras.utils.set_random_seed(SEED)
print('Python:', sys.executable)
print('TensorFlow:', tf.__version__)
print('Available GPUs:', tf.config.list_physical_devices('GPU'))
print('Mode:', 'FULL DATASET' if SAMPLE_LIMIT is None else 'PIPELINE CHECK')


In [ ]:
# To specify the dataset manually, use Path('/kaggle/input/competitions/asl-signs').
# Search only a few directory levels to avoid traversing every parquet file.
if DATA_DIR is None:
    candidates = []
    for pattern in ('train.csv', '*/train.csv', '*/*/train.csv', '*/*/*/train.csv'):
        for csv_path in INPUT_DIR.glob(pattern):
            if (csv_path.parent / 'sign_to_prediction_index_map.json').is_file():
                candidates.append(csv_path.parent)
    candidates = sorted(set(candidates))
    if not candidates:
        raise FileNotFoundError(
            'ASL competition data was not found. In your Kaggle notebook, use Add Input to add '
            'Google - Isolated Sign Language Recognition (asl-signs), then rerun this cell.'
        )
    if len(candidates) > 1:
        raise RuntimeError(f'Multiple dataset directories found; set DATA_DIR explicitly above: {candidates}')
    DATA_DIR = candidates[0]
print('Data directory:', DATA_DIR)

table = pd.read_csv(DATA_DIR / 'train.csv')
labels = json.loads((DATA_DIR / 'sign_to_prediction_index_map.json').read_text())
required = {'path', 'sign', 'participant_id'}
if not required.issubset(table.columns):
    raise ValueError(f'train.csv is missing required columns: {required - set(table.columns)}')
if table.empty:
    raise ValueError('train.csv contains no samples')
if sorted(labels.values()) != list(range(len(labels))):
    raise ValueError('Label indices must be contiguous and start at zero')
if not table['sign'].isin(labels).all():
    raise ValueError('train.csv contains labels missing from the official label mapping')
print('Samples:', len(table))
print('Classes:', len(labels))
print('Signers:', table['participant_id'].nunique())
display(table.head())
display(table.groupby('participant_id').size().rename('samples').to_frame())


## Split by signer before sampling

The split uses all metadata first. A signer belongs to only one partition. In pipeline-check mode, samples are then selected independently from each partition. Many classes may be absent in a small run, so its accuracy should not be reported as a full-task result.


In [ ]:
if table['participant_id'].nunique() < 2:
    raise ValueError('At least two distinct signers are required.')
splitter = GroupShuffleSplit(n_splits=1, test_size=VALIDATION_FRACTION, random_state=SEED)
train_indices, val_indices = next(splitter.split(table, groups=table['participant_id']))
train_rows = table.iloc[train_indices].copy()
val_rows = table.iloc[val_indices].copy()
train_signers = set(train_rows['participant_id'])
val_signers = set(val_rows['participant_id'])
assert train_signers.isdisjoint(val_signers), 'Signer leakage detected.'

if SAMPLE_LIMIT is not None:
    train_count = max(1, int(SAMPLE_LIMIT * (1 - VALIDATION_FRACTION)))
    val_count = max(1, SAMPLE_LIMIT - train_count)
    train_rows = train_rows.sample(n=min(train_count, len(train_rows)), random_state=SEED)
    val_rows = val_rows.sample(n=min(val_count, len(val_rows)), random_state=SEED)
train_rows = train_rows.reset_index(drop=True)
val_rows = val_rows.reset_index(drop=True)
N_CLASSES = len(labels)
print('Training samples:', len(train_rows), '| classes:', train_rows['sign'].nunique())
print('Validation samples:', len(val_rows), '| classes:', val_rows['sign'].nunique())
print('Training signers:', sorted(train_signers))
print('Validation signers:', sorted(val_signers))
print('Validation classes absent from selected training data:',
      len(set(val_rows['sign']) - set(train_rows['sign'])))

run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN_DIR = OUTPUT_ROOT / ('cnn_body_local_' + run_id)
RUN_DIR.mkdir(parents=True, exist_ok=False)
config = dict(seed=SEED, sample_limit=SAMPLE_LIMIT, epochs=EPOCHS,
              frames=N_FRAMES, batch_size=BATCH_SIZE,
              validation_fraction=VALIDATION_FRACTION, data_dir=str(DATA_DIR),
              tensorflow_version=tf.__version__, n_classes=N_CLASSES,
              train_samples=len(train_rows), validation_samples=len(val_rows),
              train_signers=[str(x) for x in sorted(train_signers)],
              validation_signers=[str(x) for x in sorted(val_signers)],
              feature_version='hands_body_plus_local_shape_v3', n_features=N_FEATURES)
(RUN_DIR / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
(RUN_DIR / 'labels.json').write_text(json.dumps(labels, indent=2), encoding='utf-8')
train_rows.to_csv(RUN_DIR / 'train_split.csv', index=False)
val_rows.to_csv(RUN_DIR / 'validation_split.csv', index=False)
print('Run directory:', RUN_DIR)


## Two complementary representations

The first 126 features are identical to notebook 03: body-relative hand xy coordinates followed by their validity masks. Body normalization prefers a clip-level shoulder midpoint and width, with face and hand fallbacks.

For each hand in each retained frame, append:

- Local xy coordinates centered at its wrist (hand point 0). If the wrist is missing, use the centroid of the visible points in that hand for that frame.
- Coordinates scaled by the root-mean-square distance of visible points from that anchor. This removes hand translation and a common xy scale, but preserves orientation.
- A local validity mask. A missing hand or a hand with degenerate scale produces zeros and a zero local mask.

The final order is **84 body xy + 42 body masks + 84 local xy + 42 local masks = 252 features**. Frame removal and resampling match notebook 03. This includes shape cues but does not add z coordinates or facial expression inputs.


In [ ]:
def load_frames(path):
    import numpy as np
    import pandas as pd
    data = pd.read_parquet(path, columns=["x", "y", "z"]).to_numpy(dtype=np.float32)
    if len(data) == 0 or len(data) % 543:
        raise ValueError(f"Invalid landmark rows in {path}: {len(data)}")
    return data.reshape(-1, 543, 3)

def clip_reference(frames, hands, valid):
    """Return a stable origin, positive scalar scale, and reference source."""
    xy = np.asarray(frames[..., :2], dtype=np.float64)
    shoulders = xy[:, [489 + 11, 489 + 12], :]
    shoulder_valid = np.isfinite(shoulders).all(axis=(1, 2))
    pairs = shoulders[shoulder_valid]
    if len(pairs):
        widths = np.linalg.norm(pairs[:, 0] - pairs[:, 1], axis=1)
        usable = np.isfinite(widths) & (widths > 1e-6)
        if usable.any():
            center = np.median(pairs[usable].mean(axis=1), axis=0)
            scale = float(np.median(widths[usable]))
            return center, max(scale, 1e-6), 'shoulders'

    face = xy[:, :468, :]
    centers, spreads = [], []
    for points in face:
        points = points[np.isfinite(points).all(axis=1)]
        if len(points) < 3:
            continue
        center = np.median(points, axis=0)
        spread = float(np.sqrt(np.mean(np.sum((points - center) ** 2, axis=1))))
        if np.isfinite(spread) and spread > 1e-6:
            centers.append(center)
            spreads.append(spread)
    if centers:
        return np.median(centers, axis=0), float(np.median(spreads)), 'face'

    points = np.asarray(hands[valid], dtype=np.float64)
    if len(points):
        center = np.median(points, axis=0)
        spread = float(np.sqrt(np.mean(np.sum((points - center) ** 2, axis=1))))
        return center, max(spread, 1e-6), 'hands_fallback'
    return np.zeros(2), 1.0, 'no_hands'


def preprocess_body(frames, length, return_reference=False):
    """Normalize hand xy by a clip-level body reference; keep 126 features."""
    hands = frames[:, list(range(468, 489)) + list(range(522, 543)), :2]
    valid = np.isfinite(hands).all(axis=-1)
    useful = valid.any(axis=1)
    hands, valid = hands[useful], valid[useful]
    if not len(hands):
        result = np.zeros((length, 126), dtype=np.float32)
        return (result, 'no_hands') if return_reference else result
    center, scale, source = clip_reference(frames[useful], hands, valid)
    safe = np.where(valid[..., None], hands, center)
    normalized = np.where(valid[..., None], (safe - center) / scale, 0)
    features = np.concatenate([normalized.reshape(len(hands), 84), valid], axis=1)
    indices = np.rint(np.linspace(0, len(features) - 1, length)).astype(int)
    result = features[indices].astype(np.float32)
    return (result, source) if return_reference else result


def preprocess(frames, length, return_reference=False):
    """Concatenate the unchanged body representation and local hand shape."""
    body, reference = preprocess_body(frames, length, return_reference=True)
    hands = frames[:, list(range(468, 489)) + list(range(522, 543)), :2]
    visible = np.isfinite(hands).all(axis=-1)
    useful = visible.any(axis=1)
    hands, visible = hands[useful], visible[useful]
    if not len(hands):
        result = np.zeros((length, 252), dtype=np.float32)
        return (result, reference) if return_reference else result
    local = np.zeros_like(hands, dtype=np.float64)
    local_valid = np.zeros_like(visible)
    for start in (0, 21):
        points = np.asarray(hands[:, start:start + 21], dtype=np.float64)
        valid = visible[:, start:start + 21]
        count = np.maximum(valid.sum(axis=1), 1)[:, None]
        safe = np.where(valid[..., None], points, 0)
        centroid = safe.sum(axis=1) / count
        anchor = np.where(valid[:, 0, None], safe[:, 0], centroid)
        centered = np.where(valid[..., None], safe - anchor[:, None, :], 0)
        scale = np.sqrt(np.sum(centered ** 2, axis=(1, 2)) / count[:, 0])
        usable = np.isfinite(scale) & (scale > 1e-6)
        mask = valid & usable[:, None]
        normalized = centered / np.maximum(scale[:, None, None], 1e-6)
        local[:, start:start + 21] = np.where(mask[..., None], normalized, 0)
        local_valid[:, start:start + 21] = mask
    indices = np.rint(np.linspace(0, len(hands) - 1, length)).astype(int)
    shape_features = np.concatenate(
        [local.reshape(len(hands), 84), local_valid], axis=1)[indices]
    result = np.concatenate([body, shape_features], axis=1).astype(np.float32)
    return (result, reference) if return_reference else result


In [ ]:
def prepare_partition(rows, name):
    feature_path = RUN_DIR / f'{name}_features.npy'
    x = np.lib.format.open_memmap(
        feature_path, mode='w+', dtype=np.float32,
        shape=(len(rows), N_FRAMES, N_FEATURES))
    y = rows['sign'].map(labels).to_numpy(dtype=np.int64)
    empty_count = 0
    reference_counts = {}
    for index, row in enumerate(rows.itertuples(index=False)):
        features, reference = preprocess(
            load_frames(DATA_DIR / row.path), N_FRAMES, return_reference=True)
        reference_counts[reference] = reference_counts.get(reference, 0) + 1
        if not np.isfinite(features).all():
            raise ValueError(f'Non-finite features for {row.path}')
        x[index] = features
        empty_count += int(not features[:, 84:126].any())
        if (index + 1) % 500 == 0 or index + 1 == len(rows):
            print(f'{name}: {index + 1}/{len(rows)}', flush=True)
    x.flush()
    del x
    np.save(RUN_DIR / f'{name}_labels.npy', y)
    print(f'{name}: {empty_count} sequences have no valid hand points.')
    (RUN_DIR / f'{name}_reference_counts.json').write_text(
        json.dumps(reference_counts, indent=2), encoding='utf-8')
    print('Normalization references:', reference_counts)
    return np.load(feature_path, mmap_mode='r'), y

estimated_bytes = (len(train_rows) + len(val_rows)) * N_FRAMES * N_FEATURES * 4
print(f'Feature cache estimate: {estimated_bytes / 1024**3:.2f} GiB')
x_train, y_train = prepare_partition(train_rows, 'train')
x_val, y_val = prepare_partition(val_rows, 'validation')
print('Train:', x_train.shape, '| validation:', x_val.shape)


In [ ]:
def make_dataset(x, y, shuffle):
    rng = np.random.default_rng(SEED)
    def batches():
        indices = rng.permutation(len(y)) if shuffle else np.arange(len(y))
        for start in range(0, len(y), BATCH_SIZE):
            batch_indices = indices[start:start + BATCH_SIZE]
            yield np.asarray(x[batch_indices], dtype=np.float32), y[batch_indices]
    dataset = tf.data.Dataset.from_generator(
        batches,
        output_signature=(
            tf.TensorSpec((None, N_FRAMES, N_FEATURES), tf.float32),
            tf.TensorSpec((None,), tf.int64)))
    return dataset.apply(tf.data.experimental.assert_cardinality(
        math.ceil(len(y) / BATCH_SIZE))).prefetch(1)

train_ds = make_dataset(x_train, y_train, shuffle=True)
val_ds = make_dataset(x_val, y_val, shuffle=False)
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)
model = tf.keras.Sequential([
    tf.keras.layers.Input((N_FRAMES, N_FEATURES)),
    tf.keras.layers.Conv1D(96, 5, padding='same', activation='relu'),
    tf.keras.layers.Conv1D(128, 3, padding='same', activation='relu'),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(N_CLASSES, activation='softmax'),
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()
config['model_parameters'] = int(model.count_params())
(RUN_DIR / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
print(f'Uniform random accuracy reference: {1 / N_CLASSES:.2%}')


## Train

Executing the next cell starts full-dataset training with the default configuration. The best checkpoint is selected by validation accuracy. Early stopping may finish before 20 epochs if validation accuracy stops improving.


In [ ]:
checkpoint_path = RUN_DIR / 'best.keras'
history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(
            str(checkpoint_path), monitor='val_accuracy', mode='max', save_best_only=True),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy', mode='max', patience=5, restore_best_weights=True),
        tf.keras.callbacks.CSVLogger(str(RUN_DIR / 'training_log.csv')),
    ])
(RUN_DIR / 'history.json').write_text(
    json.dumps({k: [float(v) for v in values] for k, values in history.history.items()}, indent=2),
    encoding='utf-8')
print('Saved best model:', checkpoint_path)


In [ ]:
best_model = tf.keras.models.load_model(checkpoint_path)
scores = {k: float(v) for k, v in best_model.evaluate(val_ds, return_dict=True).items()}
# Keep prediction order identical to validation_split.csv.
prediction_ds = val_ds.map(lambda features, targets: features)
probabilities = best_model.predict(prediction_ds)
predictions = probabilities.argmax(axis=1)
assert len(predictions) == len(y_val)
metrics = dict(scores, pipeline_check=SAMPLE_LIMIT is not None,
               validation_samples=len(y_val),
               best_epoch=int(np.argmax(history.history['val_accuracy']) + 1))
(RUN_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
inverse_labels = {index: sign for sign, index in labels.items()}
class_names = [inverse_labels[i] for i in range(N_CLASSES)]
report = classification_report(
    y_val, predictions, labels=list(range(N_CLASSES)),
    target_names=class_names, output_dict=True, zero_division=0)
pd.DataFrame(report).T.to_csv(RUN_DIR / 'classification_report.csv')
results = val_rows[['path', 'participant_id', 'sign']].copy()
results['predicted_sign'] = [inverse_labels[int(i)] for i in predictions]
results['confidence'] = probabilities.max(axis=1)
results.to_csv(RUN_DIR / 'validation_predictions.csv', index=False)
print(json.dumps(metrics, indent=2))
if SAMPLE_LIMIT is not None:
    print('PIPELINE CHECK ONLY: rerun with SAMPLE_LIMIT=None for a full-task baseline.')
display(results.head(10))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for key, title, axis in [('loss', 'Loss', axes[0]), ('accuracy', 'Accuracy', axes[1])]:
    epoch_numbers = range(1, len(history.history[key]) + 1)
    axis.plot(epoch_numbers, history.history[key], label='train')
    axis.plot(epoch_numbers, history.history['val_' + key], label='validation')
    axis.set(title=title, xlabel='Epoch', ylabel=title)
    axis.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / 'learning_curves.png', dpi=150)
plt.show()

matrix = confusion_matrix(y_val, predictions, labels=list(range(N_CLASSES)))
np.save(RUN_DIR / 'confusion_matrix.npy', matrix)
np.fill_diagonal(matrix, 0)
confusions = [(int(matrix[i, j]), class_names[i], class_names[j])
              for i, j in zip(*np.nonzero(matrix))]
confusions.sort(reverse=True)
display(pd.DataFrame(confusions[:15], columns=['count', 'true_sign', 'predicted_sign']))
print('Artifacts saved to:', RUN_DIR)


In [ ]:
# Reuse a compatible saved baseline if available; never compare different splits.
if BASELINE_RUN_DIR is None:
    candidates = []
    for path in OUTPUT_ROOT.glob('cnn_*'):
        if path == RUN_DIR or not (path / 'config.json').is_file():
            continue
        baseline_config = json.loads((path / 'config.json').read_text())
        if (baseline_config.get('feature_version') == 'hands_xy_mask_clip_body_reference_v2'
                and baseline_config.get('sample_limit') == SAMPLE_LIMIT
                and (path / 'metrics.json').is_file()):
            candidates.append(path)
    if candidates:
        BASELINE_RUN_DIR = sorted(candidates)[-1]

if BASELINE_RUN_DIR is None:
    print('No saved baseline was found. Set BASELINE_RUN_DIR to compare later.')
else:
    BASELINE_RUN_DIR = Path(BASELINE_RUN_DIR)
    previous_config = json.loads((BASELINE_RUN_DIR / 'config.json').read_text())
    fields = ['seed', 'sample_limit', 'epochs', 'frames', 'batch_size',
              'validation_fraction', 'n_classes', 'tensorflow_version']
    mismatches = [key for key in fields if previous_config.get(key) != config.get(key)]
    if previous_config.get('feature_version') != 'hands_xy_mask_clip_body_reference_v2':
        mismatches.append('baseline feature_version')
    for name, current in [('train_split.csv', train_rows), ('validation_split.csv', val_rows)]:
        previous = pd.read_csv(BASELINE_RUN_DIR / name)
        fields_to_check = ['path', 'sign', 'participant_id']
        if not previous[fields_to_check].astype(str).equals(current[fields_to_check].astype(str)):
            mismatches.append(name)
    previous_labels = json.loads((BASELINE_RUN_DIR / 'labels.json').read_text())
    if previous_labels != labels:
        mismatches.append('label mapping')
    if mismatches:
        print('Comparison skipped: incompatible settings or splits:', mismatches)
    else:
        previous_metrics = json.loads((BASELINE_RUN_DIR / 'metrics.json').read_text())
        comparison = pd.DataFrame([
            dict(experiment='body_reference_baseline', accuracy=previous_metrics['accuracy'],
                 loss=previous_metrics['loss'], run_dir=str(BASELINE_RUN_DIR)),
            dict(experiment='body_plus_local_shape', accuracy=metrics['accuracy'],
                 loss=metrics['loss'], run_dir=str(RUN_DIR)),
        ])
        comparison.to_csv(RUN_DIR / 'baseline_comparison.csv', index=False)
        display(comparison)
        delta = 100 * (metrics['accuracy'] - previous_metrics['accuracy'])
        print(f'Validation accuracy change: {delta:+.2f} percentage points')
        print('Input size and parameter count differ; this is not a capacity-matched comparison.')
        old_report = pd.read_csv(BASELINE_RUN_DIR / 'classification_report.csv', index_col=0)
        new_report = pd.DataFrame(report).T
        recalls = pd.DataFrame({
            'body_reference_recall': old_report.loc[class_names, 'recall'],
            'combined_recall': new_report.loc[class_names, 'recall'],
            'support': new_report.loc[class_names, 'support'],
        })
        recalls['recall_change'] = recalls['combined_recall'] - recalls['body_reference_recall']
        recalls.to_csv(RUN_DIR / 'per_class_comparison.csv')
        focus = ['mouth', 'lips', 'chin', 'nose', 'tongue']
        display(recalls.loc[[name for name in focus if name in recalls.index]])


## Interpret the experiment

Compare the best checkpoint against notebook 03 using the same full-data partitions and training settings. The comparison checks both sample lists, the official label mapping, and key settings. If baseline artifacts are missing or incompatible, it skips the comparison instead of substituting screenshot values.

Inspect overall accuracy, validation loss, per-class recall and support, and the train/validation gap. Keep the original 46.2752% body-reference model until this run has been evaluated. A single run does not establish statistical significance, and increased input size also increases model capacity.

Save `best.keras`, `labels.json`, `config.json`, metrics, learning curves, reference counts, and comparison reports before ending the Kaggle session. This model accepts preprocessed `(32, 252)` input and is not a competition-ready TFLite submission.
